# 309 — Phenotype Sensitivity Analysis

## Objective

Evaluate whether the resistance-like pharmacological phenotype selected in notebook 307 is robust to alternative, methodologically justified phenotype representations within the frozen cell-line modeling cohort.

The primary phenotype remains the model-level median raw LN_IC50 selected in notebook 307. This notebook does not perform a new phenotype-selection step.

Specifically, this notebook assesses:

- sensitivity to alternative pharmacological response metrics;
- sensitivity to model-level aggregation strategy;
- lineage-associated phenotype structure;
- concordance after explicit lineage adjustment;
- uncertainty under lineage-stratified resampling;
- sensitivity to model-level drug-response coverage.

The objective is to determine whether the pharmacological representation used for downstream cell-line program discovery is excessively dependent on the specific phenotype definition selected upstream.

## Scope

This notebook is part of the Cell-Line Discovery Layer.

It does not reconstruct the modeling cohort, repeat pharmacology integration or quality control, redefine the primary phenotype, perform transcriptomic feature selection, discover transcriptomic programs, or compare cell-line results with TCGA tumor programs.

Transcriptomic program discovery and robustness analyses will be performed in notebooks 310 and 311.

Cross-system comparison with independently discovered tumor programs belongs to Phase 4.

## Conceptual distinction

Notebook 307 answers:

> Which pharmacology-derived representation should serve as the primary resistance-like phenotype?

Notebook 309 answers:

> Are the properties of that selected phenotype robust to reasonable alternative representations and lineage-aware sensitivity analyses?

This distinction is important because phenotype selection and phenotype sensitivity assessment represent different analytical steps.

The selected phenotype remains fixed during this notebook. Alternative representations are used only to characterize sensitivity and uncertainty.

## Expected outputs

This notebook will generate:

```text
data/interim/pharmacology/309_phenotype_sensitivity_representations.parquet
data/interim/pharmacology/309_phenotype_sensitivity_summary.csv
```

The first artifact will contain the model-level phenotype representations required for sensitivity analyses.

The second artifact will summarize lineage dependence, concordance with the primary phenotype, lineage-adjusted concordance, resampling uncertainty, and drug-response coverage sensitivity.

## Methodological note

All comparisons are performed on the frozen model universe defined through notebook 308.

Lineage is treated explicitly as a major biological structure rather than removed indiscriminately from the primary phenotype.

Alternative pharmacological metrics and lineage-adjusted representations are sensitivity analyses only and will not replace the selected median raw LN_IC50 phenotype based on downstream performance.

The resistance-like phenotype represents relative baseline drug insensitivity in pharmacogenomic screening data. It should not be interpreted as clinical drug resistance, acquired resistance, treatment failure, or a causal biological mechanism.

---

In [1]:
# =============================================================================
# Imports
# =============================================================================

from pancancer_epigenetics.utils.paths import (Paths, project_relative_path)

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

PHARMACOLOGY_DIR = Paths.pharmacology

PHENOTYPE_PATH = (
    PHARMACOLOGY_DIR / "307_model_level_phenotype.parquet"
)

COHORT_PATH = (
    PHARMACOLOGY_DIR / "308_analysis_cohort.csv"
)

DRUG_RESPONSE_PATH = (
    PHARMACOLOGY_DIR / "306_gdsc_drug_response.csv"
)

OUTPUT_DIR = PHARMACOLOGY_DIR

In [3]:
# =============================================================================
# Load phenotype-sensitivity inputs
# =============================================================================

phenotype = pd.read_parquet(PHENOTYPE_PATH)

analysis_cohort = pd.read_csv(COHORT_PATH)

drug_response = pd.read_csv(DRUG_RESPONSE_PATH)

print("Phenotype table :", phenotype.shape)
print("Analysis cohort :", analysis_cohort.shape)
print("Drug response   :", drug_response.shape)

Phenotype table : (713, 7)
Analysis cohort : (713, 9)
Drug response   : (242036, 9)


In [4]:
# =============================================================================
# Construct model-level sensitivity table
# =============================================================================

sensitivity_representations = phenotype.merge(
    analysis_cohort[
        [
            "ModelID",
            "SangerModelID",
            "OncotreeLineage",
        ]
    ],
    on="ModelID",
    how="left",
)

In [5]:
# =============================================================================
# Compute model-level drug-response coverage
# =============================================================================

drug_coverage = (
    drug_response
    .groupby("SANGER_MODEL_ID")
    .agg(
        n_drug_response_measurements=("DRUG_ID", "size"),
        n_unique_drug_ids=("DRUG_ID", "nunique"),
    )
    .reset_index()
)

In [6]:
# =============================================================================
# Integrate drug-response coverage
# =============================================================================

sensitivity_representations = (
    sensitivity_representations
    .merge(
        drug_coverage,
        left_on="SangerModelID",
        right_on="SANGER_MODEL_ID",
        how="left",
    )
    .drop(columns="SANGER_MODEL_ID")
)

In [7]:
# =============================================================================
# Summarize model-level drug-response coverage
# =============================================================================

coverage_summary = (
    sensitivity_representations[
        [
            "n_drug_response_measurements",
            "n_unique_drug_ids",
        ]
    ]
    .describe()
    .loc[
        [
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

coverage_summary

,n_drug_response_measurements,n_unique_drug_ids
min,12.0,12.0
25%,251.0,251.0
50%,279.0,279.0
75%,280.0,280.0
max,295.0,295.0


In [8]:
# =============================================================================
# Compare drug-response coverage measures
# =============================================================================

coverage_measures_identical = (
    sensitivity_representations["n_drug_response_measurements"]
    .eq(sensitivity_representations["n_unique_drug_ids"])
    .all()
)

print("Coverage measures identical:", coverage_measures_identical)

Coverage measures identical: True


In [9]:
# =============================================================================
# Remove redundant coverage measure
# =============================================================================

sensitivity_representations = (
    sensitivity_representations
    .drop(columns="n_drug_response_measurements")
)

In [10]:
# =============================================================================
# Define phenotype sensitivity panel
# =============================================================================

PRIMARY_PHENOTYPE = "selected_phenotype"

CORE_SENSITIVITY_PHENOTYPES = [
    "mean_ln_ic50",
    "mean_auc",
    "mean_z_score",
]

SECONDARY_SENSITIVITY_PHENOTYPES = [
    "median_auc",
    "median_z_score",
]

PHENOTYPE_REPRESENTATIONS = [
    PRIMARY_PHENOTYPE,
    *CORE_SENSITIVITY_PHENOTYPES,
    *SECONDARY_SENSITIVITY_PHENOTYPES,
]

In [11]:
# =============================================================================
# Quantify model-level lineage-associated variance
# =============================================================================

lineage_eta_squared = {}

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    values = sensitivity_representations[phenotype_name]
    grand_mean = values.mean()

    total_ss = ((values - grand_mean) ** 2).sum()

    lineage_means = (
        sensitivity_representations
        .groupby("OncotreeLineage")[phenotype_name]
        .transform("mean")
    )

    between_ss = ((lineage_means - grand_mean) ** 2).sum()

    lineage_eta_squared[phenotype_name] = between_ss / total_ss

lineage_association_summary = (
    pd.Series(
        lineage_eta_squared,
        name="lineage_eta_squared",
    )
    .rename_axis("phenotype_representation")
    .reset_index()
)

lineage_association_summary

,phenotype_representation,lineage_eta_squared
0,selected_phenotype,0.393189
1,mean_ln_ic50,0.419183
2,mean_auc,0.367512
3,mean_z_score,0.422613
4,median_auc,0.188762
5,median_z_score,0.416977


In [12]:
# =============================================================================
# Construct lineage-adjusted phenotype representations
# =============================================================================

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    sensitivity_representations[
        f"{phenotype_name}_lineage_residual"
    ] = (
        sensitivity_representations[phenotype_name]
        - sensitivity_representations
        .groupby("OncotreeLineage")[phenotype_name]
        .transform("mean")
    )

In [13]:
# =============================================================================
# Compute raw phenotype concordance
# =============================================================================

raw_concordance = []

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    if phenotype_name == PRIMARY_PHENOTYPE:
        continue

    rho, _ = spearmanr(
        sensitivity_representations[PRIMARY_PHENOTYPE],
        sensitivity_representations[phenotype_name],
    )

    raw_concordance.append(
        {
            "phenotype_representation": phenotype_name,
            "raw_spearman_vs_primary": rho,
        }
    )

raw_concordance = pd.DataFrame(raw_concordance)

raw_concordance

,phenotype_representation,raw_spearman_vs_primary
0,mean_ln_ic50,0.988848
1,mean_auc,0.837331
2,mean_z_score,0.982176
3,median_auc,0.529273
4,median_z_score,0.980391


In [14]:
print(raw_concordance)

  phenotype_representation  raw_spearman_vs_primary
0             mean_ln_ic50                 0.988848
1                 mean_auc                 0.837331
2             mean_z_score                 0.982176
3               median_auc                 0.529273
4           median_z_score                 0.980391


In [15]:
# =============================================================================
# Compute lineage-adjusted phenotype concordance
# =============================================================================

lineage_adjusted_concordance = []

primary_residual = f"{PRIMARY_PHENOTYPE}_lineage_residual"

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    if phenotype_name == PRIMARY_PHENOTYPE:
        continue

    alternative_residual = f"{phenotype_name}_lineage_residual"

    rho, _ = spearmanr(
        sensitivity_representations[primary_residual],
        sensitivity_representations[alternative_residual],
    )

    lineage_adjusted_concordance.append(
        {
            "phenotype_representation": phenotype_name,
            "lineage_adjusted_spearman_vs_primary": rho,
        }
    )

lineage_adjusted_concordance = pd.DataFrame(
    lineage_adjusted_concordance
)

lineage_adjusted_concordance

,phenotype_representation,lineage_adjusted_spearman_vs_primary
0,mean_ln_ic50,0.982548
1,mean_auc,0.774458
2,mean_z_score,0.973620
3,median_auc,0.502363
4,median_z_score,0.969455


In [16]:
# =============================================================================
# Define lineage residualization
# =============================================================================

def residualize_by_lineage(data, phenotype_name):
    return (
        data[phenotype_name]
        - data
        .groupby("OncotreeLineage")[phenotype_name]
        .transform("mean")
    )

In [17]:
# =============================================================================
# Define lineage-stratified model bootstrap
# =============================================================================

N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20260812

rng = np.random.default_rng(BOOTSTRAP_SEED)


def stratified_model_bootstrap(data, rng):
    sampled_lineages = []

    for _, lineage_data in data.groupby(
        "OncotreeLineage",
        sort=False,
    ):
        sampled_indices = rng.choice(
            lineage_data.index.to_numpy(),
            size=len(lineage_data),
            replace=True,
        )

        sampled_lineages.append(
            data.loc[sampled_indices]
        )

    return pd.concat(
        sampled_lineages,
        ignore_index=True,
    )

In [18]:
# =============================================================================
# Run lineage-stratified bootstrap
# =============================================================================

bootstrap_results = []

for bootstrap_iteration in range(N_BOOTSTRAP):
    bootstrap_sample = stratified_model_bootstrap(
        sensitivity_representations,
        rng,
    )

    primary_residual = residualize_by_lineage(
        bootstrap_sample,
        PRIMARY_PHENOTYPE,
    )

    for phenotype_name in PHENOTYPE_REPRESENTATIONS:
        if phenotype_name == PRIMARY_PHENOTYPE:
            continue

        raw_rho, _ = spearmanr(
            bootstrap_sample[PRIMARY_PHENOTYPE],
            bootstrap_sample[phenotype_name],
        )

        alternative_residual = residualize_by_lineage(
            bootstrap_sample,
            phenotype_name,
        )

        adjusted_rho, _ = spearmanr(
            primary_residual,
            alternative_residual,
        )

        bootstrap_results.append(
            {
                "bootstrap_iteration": bootstrap_iteration,
                "phenotype_representation": phenotype_name,
                "raw_spearman": raw_rho,
                "lineage_adjusted_spearman": adjusted_rho,
            }
        )

bootstrap_results = pd.DataFrame(bootstrap_results)

In [19]:
# =============================================================================
# Summarize bootstrap concordance uncertainty
# =============================================================================

bootstrap_summary = (
    bootstrap_results
    .groupby("phenotype_representation")
    .agg(
        raw_spearman_bootstrap_median=(
            "raw_spearman",
            "median",
        ),
        raw_spearman_ci_lower=(
            "raw_spearman",
            lambda x: x.quantile(0.025),
        ),
        raw_spearman_ci_upper=(
            "raw_spearman",
            lambda x: x.quantile(0.975),
        ),
        lineage_adjusted_bootstrap_median=(
            "lineage_adjusted_spearman",
            "median",
        ),
        lineage_adjusted_ci_lower=(
            "lineage_adjusted_spearman",
            lambda x: x.quantile(0.025),
        ),
        lineage_adjusted_ci_upper=(
            "lineage_adjusted_spearman",
            lambda x: x.quantile(0.975),
        ),
    )
    .reset_index()
)

bootstrap_summary

,phenotype_representation,raw_spearman_bootstrap_median,raw_spearman_ci_lower,raw_spearman_ci_upper,lineage_adjusted_bootstrap_median,lineage_adjusted_ci_lower,lineage_adjusted_ci_upper
0,mean_auc,0.837315,0.810742,0.860342,0.772408,0.733188,0.804094
1,mean_ln_ic50,0.988605,0.986319,0.990592,0.981880,0.977858,0.985135
2,mean_z_score,0.982349,0.974528,0.987081,0.973410,0.963333,0.979387
3,median_auc,0.528695,0.472424,0.582346,0.500425,0.439230,0.555940
4,median_z_score,0.980489,0.973371,0.985026,0.969033,0.960704,0.975049


In [20]:
# =============================================================================
# Assess drug-coverage sensitivity of the primary phenotype
# =============================================================================

coverage_lineage_residual = residualize_by_lineage(
    sensitivity_representations,
    "n_unique_drug_ids",
)

raw_coverage_rho, _ = spearmanr(
    sensitivity_representations[PRIMARY_PHENOTYPE],
    sensitivity_representations["n_unique_drug_ids"],
)

adjusted_coverage_rho, _ = spearmanr(
    sensitivity_representations[
        f"{PRIMARY_PHENOTYPE}_lineage_residual"
    ],
    coverage_lineage_residual,
)

coverage_sensitivity = pd.DataFrame(
    {
        "comparison": [
            "raw",
            "lineage_adjusted",
        ],
        "spearman_rho": [
            raw_coverage_rho,
            adjusted_coverage_rho,
        ],
    }
)

coverage_sensitivity

,comparison,spearman_rho
0,raw,0.241556
1,lineage_adjusted,0.145902


In [21]:
# =============================================================================
# Bootstrap drug-coverage sensitivity
# =============================================================================

coverage_bootstrap_results = []
coverage_rng = np.random.default_rng(BOOTSTRAP_SEED + 1)

for bootstrap_iteration in range(N_BOOTSTRAP):
    bootstrap_sample = stratified_model_bootstrap(
        sensitivity_representations,
        coverage_rng,
    )

    primary_residual = residualize_by_lineage(
        bootstrap_sample,
        PRIMARY_PHENOTYPE,
    )

    coverage_residual = residualize_by_lineage(
        bootstrap_sample,
        "n_unique_drug_ids",
    )

    raw_rho, _ = spearmanr(
        bootstrap_sample[PRIMARY_PHENOTYPE],
        bootstrap_sample["n_unique_drug_ids"],
    )

    adjusted_rho, _ = spearmanr(
        primary_residual,
        coverage_residual,
    )

    coverage_bootstrap_results.append(
        {
            "bootstrap_iteration": bootstrap_iteration,
            "raw_spearman": raw_rho,
            "lineage_adjusted_spearman": adjusted_rho,
        }
    )

coverage_bootstrap_results = pd.DataFrame(
    coverage_bootstrap_results
)

In [22]:
# =============================================================================
# Summarize drug-coverage bootstrap uncertainty
# =============================================================================

coverage_bootstrap_summary = pd.DataFrame(
    {
        "comparison": [
            "raw",
            "lineage_adjusted",
        ],
        "bootstrap_median": [
            coverage_bootstrap_results["raw_spearman"].median(),
            coverage_bootstrap_results[
                "lineage_adjusted_spearman"
            ].median(),
        ],
        "ci_lower": [
            coverage_bootstrap_results[
                "raw_spearman"
            ].quantile(0.025),
            coverage_bootstrap_results[
                "lineage_adjusted_spearman"
            ].quantile(0.025),
        ],
        "ci_upper": [
            coverage_bootstrap_results[
                "raw_spearman"
            ].quantile(0.975),
            coverage_bootstrap_results[
                "lineage_adjusted_spearman"
            ].quantile(0.975),
        ],
    }
)

coverage_bootstrap_summary

,comparison,bootstrap_median,ci_lower,ci_upper
0,raw,0.241137,0.166984,0.309852
1,lineage_adjusted,0.144995,0.073916,0.219365


In [23]:
# =============================================================================
# Assemble phenotype sensitivity summary
# =============================================================================

phenotype_sensitivity_summary = (
    lineage_association_summary
    .merge(
        raw_concordance,
        on="phenotype_representation",
        how="left",
    )
    .merge(
        lineage_adjusted_concordance,
        on="phenotype_representation",
        how="left",
    )
    .merge(
        bootstrap_summary,
        on="phenotype_representation",
        how="left",
    )
)

phenotype_sensitivity_summary.insert(
    1,
    "analysis_role",
    phenotype_sensitivity_summary["phenotype_representation"].map(
        {
            PRIMARY_PHENOTYPE: "primary",
            **{
                phenotype_name: "core_sensitivity"
                for phenotype_name in CORE_SENSITIVITY_PHENOTYPES
            },
            **{
                phenotype_name: "secondary_sensitivity"
                for phenotype_name in SECONDARY_SENSITIVITY_PHENOTYPES
            },
        }
    ),
)

phenotype_sensitivity_summary

,phenotype_representation,analysis_role,lineage_eta_squared,raw_spearman_vs_primary,lineage_adjusted_spearman_vs_primary,raw_spearman_bootstrap_median,raw_spearman_ci_lower,raw_spearman_ci_upper,lineage_adjusted_bootstrap_median,lineage_adjusted_ci_lower,lineage_adjusted_ci_upper
0,selected_phenotype,primary,0.393189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mean_ln_ic50,core_sensitivity,0.419183,0.988848,0.982548,0.988605,0.986319,0.990592,0.981880,0.977858,0.985135
2,mean_auc,core_sensitivity,0.367512,0.837331,0.774458,0.837315,0.810742,0.860342,0.772408,0.733188,0.804094
3,mean_z_score,core_sensitivity,0.422613,0.982176,0.973620,0.982349,0.974528,0.987081,0.973410,0.963333,0.979387
4,median_auc,secondary_sensitivity,0.188762,0.529273,0.502363,0.528695,0.472424,0.582346,0.500425,0.439230,0.555940
5,median_z_score,secondary_sensitivity,0.416977,0.980391,0.969455,0.980489,0.973371,0.985026,0.969033,0.960704,0.975049


In [24]:
# =============================================================================
# Integrate drug-coverage sensitivity into final summary
# =============================================================================

sensitivity_summary = (
    phenotype_sensitivity_summary
    .rename(
        columns={
            "phenotype_representation": "analysis_target",
        }
    )
)

coverage_summary_row = pd.DataFrame(
    {
        "analysis_target": ["n_unique_drug_ids"],
        "analysis_role": ["technical_sensitivity"],
        "lineage_eta_squared": [np.nan],
        "raw_spearman_vs_primary": [raw_coverage_rho],
        "lineage_adjusted_spearman_vs_primary": [
            adjusted_coverage_rho
        ],
        "raw_spearman_bootstrap_median": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"] == "raw",
                "bootstrap_median",
            ].iloc[0]
        ],
        "raw_spearman_ci_lower": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"] == "raw",
                "ci_lower",
            ].iloc[0]
        ],
        "raw_spearman_ci_upper": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"] == "raw",
                "ci_upper",
            ].iloc[0]
        ],
        "lineage_adjusted_bootstrap_median": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"]
                == "lineage_adjusted",
                "bootstrap_median",
            ].iloc[0]
        ],
        "lineage_adjusted_ci_lower": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"]
                == "lineage_adjusted",
                "ci_lower",
            ].iloc[0]
        ],
        "lineage_adjusted_ci_upper": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"]
                == "lineage_adjusted",
                "ci_upper",
            ].iloc[0]
        ],
    }
)

sensitivity_summary = pd.concat(
    [
        sensitivity_summary,
        coverage_summary_row,
    ],
    ignore_index=True,
)

sensitivity_summary

,analysis_target,analysis_role,lineage_eta_squared,raw_spearman_vs_primary,lineage_adjusted_spearman_vs_primary,raw_spearman_bootstrap_median,raw_spearman_ci_lower,raw_spearman_ci_upper,lineage_adjusted_bootstrap_median,lineage_adjusted_ci_lower,lineage_adjusted_ci_upper
0,selected_phenotype,primary,0.393189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mean_ln_ic50,core_sensitivity,0.419183,0.988848,0.982548,0.988605,0.986319,0.990592,0.981880,0.977858,0.985135
2,mean_auc,core_sensitivity,0.367512,0.837331,0.774458,0.837315,0.810742,0.860342,0.772408,0.733188,0.804094
3,mean_z_score,core_sensitivity,0.422613,0.982176,0.973620,0.982349,0.974528,0.987081,0.973410,0.963333,0.979387
4,median_auc,secondary_sensitivity,0.188762,0.529273,0.502363,0.528695,0.472424,0.582346,0.500425,0.439230,0.555940
5,median_z_score,secondary_sensitivity,0.416977,0.980391,0.969455,0.980489,0.973371,0.985026,0.969033,0.960704,0.975049
6,n_unique_drug_ids,technical_sensitivity,NaN,0.241556,0.145902,0.241137,0.166984,0.309852,0.144995,0.073916,0.219365


In [26]:
# =============================================================================
# Save phenotype sensitivity artifacts
# =============================================================================

REPRESENTATIONS_OUTPUT_PATH = (
    OUTPUT_DIR / "309_phenotype_sensitivity_representations.parquet"
)

SUMMARY_OUTPUT_PATH = (
    OUTPUT_DIR / "309_phenotype_sensitivity_summary.csv"
)

sensitivity_representations.to_parquet(
    REPRESENTATIONS_OUTPUT_PATH,
    index=False,
)

sensitivity_summary.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
)

print("Phenotype sensitivity artifacts written")
print(f"Representations: {project_relative_path(REPRESENTATIONS_OUTPUT_PATH)}")
print(f"Summary        : {project_relative_path(SUMMARY_OUTPUT_PATH)}")

Phenotype sensitivity artifacts written
Representations: data/interim/pharmacology/309_phenotype_sensitivity_representations.parquet
Summary        : data/interim/pharmacology/309_phenotype_sensitivity_summary.csv


## Summary

The selected model-level resistance-like phenotype remained highly concordant with alternative LN_IC50 and Z_SCORE representations, including after explicit lineage adjustment.

Lineage accounted for a substantial fraction of model-level phenotype variance, confirming that downstream transcriptomic analyses must remain lineage-aware.

AUC-based representations showed lower concordance with the selected phenotype, particularly median AUC, indicating that they capture partially distinct pharmacological structure rather than constituting interchangeable phenotype definitions.

Lineage-stratified bootstrap analyses showed that these concordance patterns were stable under model-level resampling.

Drug-response coverage showed a modest positive association with the selected phenotype. This association was reduced after lineage adjustment but remained detectable, and should therefore be retained as a technical sensitivity consideration in downstream analyses.

The median raw LN_IC50 phenotype selected in notebook 307 remains the frozen primary resistance-like phenotype. No alternative representation identified in this notebook provides a methodological basis for post hoc phenotype replacement.

The phenotype-sensitivity artifacts generated here provide the input framework for transcriptomic program discovery and subsequent robustness analyses in notebooks 310 and 311.